### Logistic Regression
**Logistic Regression** is a **supervised machine learning algorithm used for classification problems**, where the output is **probability -> mapped to classes**.

> It predicts **P(Y = 1 | X)** instead of a continuous value.

### Why not Linear Regression for Classification?
##### Problem:
Linear regression outputs values like:
```
-2.3,, 1.7, 5.6
```

But classification needs:
```
0 or 1
```

##### Issue:
- Outputs can be **< 0 or > 1**
- Not interpretable as probabilities

### Solution -> Sigmoid Function
#### Sigmoid (Logistic Function):
$
\sigma(z) = \frac{1}{1 + e^{-z}}
$

Where:   
$
z = \mathbf{w}^T \mathbf{x} + b
$

##### Key Properties
- Output range: **(0, 1)**
- Converts linear output -> probability
- Smooth and differentiable (important for gradient descent)

#### Intuition
| z (input) | Output σ(z) |
| --------- | ----------- |
| -∞        | 0           |
| 0         | 0.5         |
| +∞        | 1           |


### Model Representation
#### Hypothesis Function
$
h_{\theta}(x) = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}
$

This means:
```
Probability = sigmoid(Linear Combination of features)
```

### Decision Boundary
```
if P ≥ 0.5 → Class 1
else → Class 0
```

**Decision Boundary Equation:**   
$
w^T x + b = 0
$

### Cost Function
##### Why Not MSE?
- Leads ot **non-convex loss**
- Multiple local minima

##### Log Loss / Binary Cross Entropy
- Your model predicts:   
    $h_{\theta}(x)$ ∈ [0,1] -> probability that y = 1

- True label:    
    y ∈ {0,1}

The loss for a **single example**:
- If y = 1: loss becomes −log(h(x)) -> penalizes low probabilities
- If y = 0: loss becomes −log(1−h(x)) -> penalizes high probabilities

So:

Correct confident predictions -> **low loss**
Wrong confident predictions -> **very high loss**

**Cost Function:**   
$
J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log\left(h_\theta(x^{(i)})\right) + (1 - y^{(i)}) \log\left(1 - h_\theta(x^{(i)})\right) \right]
$

where:   
$m$ : number of training examples  
$y^{(i)}$ : actual label of the $i^{th}$ example
$h_{\theta}(x^{(i)})$ : predicted probability for the $i^{th}$ example   
The summation: total loss across all samples  
The negative sign: ensures loss is **positive and minimized**

### Gradient Descent Update Rule
#### Gradients
$
\frac{\partial J}{\partial w} = \frac{1}{m} X^T (\hat{y} - y)
$

$
\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} \left( \hat{y}^{(i)} - y^{(i)} \right)
$

#### Update Rules
$
w := w - \alpha \cdot \frac{\partial J}{\partial w}
$

$
b := b - \alpha \cdot \frac{\partial J}{\partial b}
$

### Example: Problem Statement
#### Use Case: Customer Churn Prediction
You are working at a telecom company. The goal is to predict whether a customer will churn (leave the service) or stay, based on their usage and account details.

- Target variable:
    - 1 -> Churn
    - 0 -> No Churn
- Features include:
    - Tenure (months)
    - Monthly charges
    - Total charges
    - Contract type
    - Internet service type
    - Payment method
    - etc.

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import logging

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [3]:
DATA_PATH ="./data/churn_dataset.csv"
MODEL_DIR = "project-binary_logistic_regression"
MODEL_NAME = "churn_logistic_pipeline.pkl"

REQUIRED_COLUMNS = [
    "tenure", "MonthlyCharges", "TotalCharges",
    "Contract", "InternetService", "PaymentMethod", "Churn"
]

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    logger.info(f"Loading data from {path}")
    return pd.read_csv(path)

In [5]:
def validate_data(df: pd.DataFrame):
    logger.info("Validating dataset...")

    missing_cols = set(REQUIRED_COLUMNS) - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing Columns: {missing_cols}")
    
    logger.info("Dataset validation passed")

In [6]:
def preprocess_data(df: pd.DataFrame):
    logger.info("Proprocessing data...")

    df = df.copy()

    df.drop(columns=["customerID"], inplace=True, errors='ignore')

    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    df.fillna(df.median(numeric_only=True), inplace=True)

    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

    X = df.drop("Churn", axis=1)
    y = df["Churn"]

    return X, y

In [8]:
def build_pipeline(X: pd.DataFrame) -> Pipeline:
    logger.info("Building ML pipeline...")

    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
    categorical_features = X.select_dtypes(include=["object"]).columns

    numeric_pipeline = Pipeline([
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ])

    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        penalty="l2",
        C=1.0
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    return pipeline

In [9]:
def train_model(pipeline: Pipeline, X_train, y_train):
    logger.info("Training model...")
    pipeline.fit(X_train, y_train)
    return pipeline

In [10]:
def evaluate_model(model, X_test, y_test):
    logger.info("Evaluating model...")

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

    auc = roc_auc_score(y_test, y_prob)
    print(f"ROC-AUC Score: {auc:.4f}")

In [11]:
def save_model(model):
    logger.info("Saving model...")

    os.makedirs(MODEL_DIR, exist_ok=True)

    model_path = os.path.join(MODEL_DIR, MODEL_NAME)

    joblib.dump(model, model_path)

    logger.info(f"Model saved at: {model_path}")

In [12]:
def main():
    try:
        # Load
        df = load_data(DATA_PATH)

        # Validate
        validate_data(df)

        # Preprocess
        X, y = preprocess_data(df)

        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.2,
            random_state=42,
            stratify=y
        )

        # Build pipeline
        pipeline = build_pipeline(X)

        # Train
        model = train_model(pipeline, X_train, y_train)

        # Evaluate
        evaluate_model(model, X_test, y_test)

        # Save
        save_model(model)

        logger.info("Pipeline execution completed successfully")

    except Exception as e:
        logger.error(f"Error: {str(e)}")
        raise


if __name__ == "__main__":
    main()

2026-05-02 22:11:21,328 - INFO - Loading data from {path}
2026-05-02 22:11:21,342 - INFO - Validating dataset...
2026-05-02 22:11:21,343 - INFO - Dataset validation passed
2026-05-02 22:11:21,344 - INFO - Proprocessing data...
2026-05-02 22:11:21,355 - INFO - Building ML pipeline...
2026-05-02 22:11:21,356 - INFO - Training model...
/Users/jatinrokde/CodeBase/SynapseWorks/MLForge/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
2026-05-02 22:11:21,377 - INFO - Evaluating model...
2026-05-02 22:11:21,383 - INFO - Saving model...
2026-05-02 22:11:21,385 - INFO - Model saved at: project-binary_logistic_regression/churn_logistic_pipeline.pkl
2026-05-02 22:


Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.88      0.91       168
           1       0.76      0.88      0.81        72

    accuracy                           0.88       240
   macro avg       0.85      0.88      0.86       240
weighted avg       0.89      0.88      0.88       240

ROC-AUC Score: 0.9622
